In [4]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [104]:
import pandas as pd
import plotly.express as px
from tqdm import tqdm

from solar_predictor.storage import find_files_at_loc, get_location_prefix, read_csv

In [108]:
loc_lat, loc_lon = 42.0, 21.43

csvs = find_files_at_loc(get_location_prefix(loc_lat, loc_lon), "/sarah3.csv")
dfs = [read_csv(csv) for csv in tqdm(csvs, desc="Reading CSVs")]
raw_df = pd.concat(dfs, ignore_index=True)

Reading CSVs: 100%|██████████| 37/37 [02:23<00:00,  3.87s/it]


In [133]:
df = raw_df.copy()

df["datetime"] = pd.to_datetime(df["datetime"])

df["date"] = df["datetime"].dt.date
df["month"] = df["datetime"].dt.month
df["month_day"] = df["datetime"].dt.strftime("%m-%d")
df["year"] = df["datetime"].dt.year

df.head()

,timestamp,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance,datetime,date,month,month_day,year
0,631152000,0.0,0.0,0.0,0.0,1990-01-01 00:00:00+00:00,1990-01-01,1,01-01,1990
1,631153800,0.0,0.0,0.0,0.0,1990-01-01 00:30:00+00:00,1990-01-01,1,01-01,1990
2,631155600,0.0,0.0,0.0,0.0,1990-01-01 01:00:00+00:00,1990-01-01,1,01-01,1990
3,631157400,0.0,0.0,0.0,0.0,1990-01-01 01:30:00+00:00,1990-01-01,1,01-01,1990
4,631159200,0.0,0.0,0.0,0.0,1990-01-01 02:00:00+00:00,1990-01-01,1,01-01,1990


In [ ]:
daily = df.groupby("date").agg({"shortwave_radiation": "mean"}).reset_index()

fig = px.line(
    daily,
    x="date",
    y="shortwave_radiation",
    labels={"shortwave_radiation": "Shortwave Radiation (W/m²)", "date": "Date"},
    title="Daily Shortwave Radiation Over Time",
)
end = daily["date"].max()
fig.update_layout(
    xaxis={
        "rangeslider": {"visible": True},
        "range": [end - pd.DateOffset(years=5), end],
    },
)
fig

## Shortwave Radiation by Months

In [175]:
daily_month = (
    df.groupby(["month", "date"]).agg({"shortwave_radiation": "mean"}).reset_index()
)


fig = px.box(
    daily_month,
    x="month",
    y="shortwave_radiation",
    labels={"shortwave_radiation": "Shortwave Radiation (W/m²)", "month": "Month"},
    title="Daily Mean Shortwave Radiation by Month"
)
fig.update_xaxes(type="category")
fig